# Code Intelligence Industry Benchmark Design

**Status:** Approved for implementation planning by the user on 2026-08-27  
**Date:** 2026-08-27  
**Design epic:** `bd-1atpy`  
**Policy optimization:** `sol_ab7967975cd248d4`  
**Model-blocking counterfactual:** `sol_f752c72299fa45f8`

This notebook specifies a reproducible evaluation harness for SPUR code intelligence using RepoQA, CrossCodeEval, and Judging Call Graphs (JCG). It is deliberately separate from conversational `memory_eval`.

Formal cells pin the live implemented profiles:

- `relational_lia` version 1 — Z3 `qf_lia_bool_int_enum`
- `state_invariant_lia` version 1 — Z3 `qf_lia_bool_int_enum`

The notebook is the authoritative design. There is no second constraint AST or hand-authored Z3 representation.

## 1. Decision and architecture

Create a new workspace crate, `crates/spur-code-eval`, because the benchmark crosses the complete `spur-graph → spur-analyst → spur-mcp` path. Keeping it inside `spur-graph::memory_eval` would conflate conversational memory with repository intelligence and would not exercise the public MCP retrieval surface.

The accepted policy is **deterministic-first**:

1. A deterministic lane evaluates retrieval and graph correctness and may block a release.
2. A scheduled model lane evaluates downstream function reproduction and code completion.
3. Model absence or model failure cannot invalidate an otherwise complete deterministic report.
4. Model results may block releases only after an explicit policy revision and a fresh solver decision.

Data flow:

`pinned sources → isolated worktree → SPUR index → frozen retrieval/graph artifacts → suite scorers → immutable report`

Z3 Optimize selected this policy lexicographically by maximizing offline CI, failure attribution, and industry fidelity, then minimizing operational burden. The result is optimal only under that declared preference order; it is not an empirical runtime measurement.

### Architecture overview

*Illustrative architecture only. The binding release, eligibility, leakage, and artifact-order contracts remain the native NS-Mermaid cells below.*

```mermaid
flowchart LR
    subgraph Sources["Pinned public sources"]
        RQ["RepoQA<br/>descriptions + target spans"]
        CC["CrossCodeEval<br/>prefixes + hidden completions"]
        JCG["JCG<br/>annotated call-site expectations"]
    end

    subgraph Harness["crates/spur-code-eval"]
        V["Validate pins, schemas,<br/>licenses, eligibility"]
        M["Materialize isolated<br/>repository revisions"]
        Q["Leakage-safe query adapter"]
        F["Freeze rankings,<br/>contexts, call graphs"]
        S["Suite-native scorers"]
        A["Immutable run artifacts<br/>and report"]
    end

    subgraph Product["SPUR code-intelligence stack"]
        G["spur-graph<br/>multi-language fact graph"]
        AN["spur-analyst<br/>semantic + graph evidence"]
        MCP["spur-mcp<br/>code_* + KCP2"]
    end

    subgraph Lanes["Execution lanes"]
        D["Deterministic gate<br/>release blocking"]
        LLM["Scheduled model lane<br/>advisory initially"]
    end

    RQ --> V
    CC --> V
    JCG --> V
    V --> M --> G --> AN --> MCP
    M --> Q --> MCP
    MCP --> F --> S --> A
    F --> D --> A
    F --> LLM --> A
```

The deterministic lane publishes RepoQA retrieval, CrossCodeEval evidence retrieval, and JCG graph results independently. The model lane consumes frozen context read-only and cannot invalidate deterministic publication under Option A.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-EVAL-RELEASE-POLICY
@type ReleaseStatus = enum[reject, publish_deterministic, publish_full]
@input sources_pinned: Bool
@input deterministic_complete: Bool
@input deterministic_pass: Bool
@input model_complete: Bool
@input model_pass: Bool
@output status: ReleaseStatus
@requires INPUT_DOMAIN: true`"]

    REJECT["`@branch REJECT
@when not (sources_pinned and deterministic_complete and deterministic_pass)
@ensures REJECT_STATUS: status = reject`"]

    DETERMINISTIC["`@branch DETERMINISTIC
@when sources_pinned and deterministic_complete and deterministic_pass and not (model_complete and model_pass)
@ensures DETERMINISTIC_STATUS: status = publish_deterministic`"]

    FULL["`@branch FULL
@when sources_pinned and deterministic_complete and deterministic_pass and model_complete and model_pass
@ensures FULL_STATUS: status = publish_full`"]

    CHECK["`@verify RELEASE_CONSISTENT: witness consistency
@verify RELEASE_DETERMINISTIC: prove determinism
@verify RELEASE_COVERAGE: prove partition_coverage
@verify RELEASE_EXCLUSIVE: prove partition_exclusive
@verify RELEASE_STATUSES: witness each status`"]

    SPEC --> REJECT --> CHECK
    SPEC --> DETERMINISTIC --> CHECK
    SPEC --> FULL --> CHECK

## 2. Goals and non-goals

### Goals

- Measure semantic repository retrieval, cross-file evidence discovery, and call-graph correctness independently.
- Preserve upstream-native dataset meaning and scorers where they exist.
- Produce stable, inspectable rankings and graph artifacts before any downstream model runs.
- Report quality together with latency, index size, evidence bytes/tokens, answer rate, ambiguity, and unsupported coverage.
- Pin every dataset, repository revision, adapter contract, query policy, scorer, and calibration split.
- Make invalid and unsupported cases visible rather than silently changing denominators.

### Non-goals

- Replacing the existing extractor microbenchmark or rename corpus.
- Treating RepoQA, CrossCodeEval, or JCG as a conversational-memory benchmark.
- Claiming one aggregate number represents all code-intelligence capabilities.
- Using hidden completions, target names, or gold graph edges in retrieval queries.
- Requiring paid model credentials for deterministic validation or report publication.
- Adding Java or C# extractor support as part of the first benchmark implementation.

## 3. Canonical case and source contract

Every upstream item becomes a `CodeEvalCase` with:

- `suite`, `case_id`, `language`, and `contract_version`
- dataset URI, immutable revision, content hash, and license metadata
- repository URI, exact commit SHA, subdirectory, and materialization hash
- leakage-safe query input and query-policy hash
- expected target symbols, source spans, files, call sites, or derived evidence identifiers
- explicit `eligible | unsupported | invalid` status and reason
- raw upstream record retained for auditing

Repositories are materialized per pinned commit in isolated worktrees. A case cannot be scored against a mixed repository root or another case's revision. Dataset adapters validate exact source counts and hashes before indexing. Unknown fields remain in raw provenance rather than being discarded.

Language support is a manifest capability. The first full lanes target the verified SPUR intersections:

- RepoQA: every language for which the pinned SPUR extractor passes the native semantic fixture and the adapter resolves the target span.
- CrossCodeEval: Python and TypeScript initially.
- JCG: Python and JavaScript initially.

Java, C#, and any other unavailable language are reported as `unsupported`, not dropped.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-EVAL-CASE-ELIGIBILITY
@type CaseStatus = enum[invalid, unsupported, eligible]
@input source_pin_valid: Bool
@input corpus_valid: Bool
@input repository_ready: Bool
@input gold_resolved: Bool
@input language_supported: Bool
@output status: CaseStatus
@requires INPUT_DOMAIN: true`"]

    INVALID["`@branch INVALID
@when not (source_pin_valid and corpus_valid and repository_ready and gold_resolved)
@ensures INVALID_STATUS: status = invalid`"]

    UNSUPPORTED["`@branch UNSUPPORTED
@when source_pin_valid and corpus_valid and repository_ready and gold_resolved and not language_supported
@ensures UNSUPPORTED_STATUS: status = unsupported`"]

    ELIGIBLE["`@branch ELIGIBLE
@when source_pin_valid and corpus_valid and repository_ready and gold_resolved and language_supported
@ensures ELIGIBLE_STATUS: status = eligible`"]

    CHECK["`@verify ELIGIBILITY_CONSISTENT: witness consistency
@verify ELIGIBILITY_DETERMINISTIC: prove determinism
@verify ELIGIBILITY_COVERAGE: prove partition_coverage
@verify ELIGIBILITY_EXCLUSIVE: prove partition_exclusive
@verify ELIGIBILITY_STATUSES: witness each status`"]

    SPEC --> INVALID --> CHECK
    SPEC --> UNSUPPORTED --> CHECK
    SPEC --> ELIGIBLE --> CHECK

## 4. Suite adapters

### RepoQA

Input is the upstream natural-language function description without target-name leakage. Gold is the pinned target function's path, name, and source span mapped to the corresponding SPUR symbol.

Deterministic metrics are Hit@1/5/10, MRR, answer rate, evidence bytes/tokens, and query latency. The optional model lane freezes the same retrieved context, asks the pinned model to reproduce the function, and delegates scoring to the upstream best-target Tree-sitter similarity evaluator. Native scores and SPUR retrieval scores remain separate.

### CrossCodeEval

Input is the current-file prefix/prompt plus its pinned repository state. The hidden completion is never available to the retriever.

CrossCodeEval does not ship ordinary information-retrieval qrels. The adapter therefore produces a versioned `spur-derived-evidence-v1` view: identifiers in the hidden completion are resolved, for scoring only, to definitions outside the target file. The derivation stores resolver version, positive evidence spans, unresolved identifiers, and an audit trace.

Deterministic metrics are evidence Recall@1/5/10, MRR, context coverage, evidence precision by token budget, answer rate, and latency. The model lane compares no-context, lexical BM25, SPUR, and oracle-frozen contexts using upstream exact match, edit similarity, and identifier metrics.

### JCG

SPUR graph artifacts are normalized into the upstream call-site JSON contract: caller method, call-site line, declared target, and resolved targets. Official direct expectations require a direct SPUR call edge. Official indirect expectations use the pinned matcher semantics over the serialized graph.

The primary result is upstream expectation pass rate by language and feature. Additional diagnostic rows report positive-target recall and forbidden-target violations where the upstream expectation is exhaustive. Global precision is not claimed for partially annotated call sites.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-EVAL-CROSSCODE-LEAKAGE
@type EvidenceStatus = enum[invalid, eligible]
@input prefix_only_query: Bool
@input hidden_completion_in_query: Bool
@input hidden_completion_in_scoring: Bool
@input evidence_resolved: Bool
@output status: EvidenceStatus
@requires INPUT_DOMAIN: true`"]

    ELIGIBLE["`@branch ELIGIBLE
@when prefix_only_query and not hidden_completion_in_query and hidden_completion_in_scoring and evidence_resolved
@ensures ELIGIBLE_STATUS: status = eligible`"]

    INVALID["`@branch INVALID
@when not (prefix_only_query and not hidden_completion_in_query and hidden_completion_in_scoring and evidence_resolved)
@ensures INVALID_STATUS: status = invalid`"]

    CHECK["`@verify LEAKAGE_CONSISTENT: witness consistency
@verify LEAKAGE_DETERMINISTIC: prove determinism
@verify LEAKAGE_COVERAGE: prove partition_coverage
@verify LEAKAGE_EXCLUSIVE: prove partition_exclusive
@verify LEAKAGE_STATUSES: witness each status`"]

    SPEC --> ELIGIBLE --> CHECK
    SPEC --> INVALID --> CHECK

## 5. Baselines, metrics, and aggregation

All retrievers receive the same leakage-safe query and the same pinned corpus:

1. exact identifier lookup, only when the query legitimately contains an identifier
2. lexical BM25
3. SPUR semantic evidence pack and exact graph follow-up
4. oracle evidence, used only as an upper bound
5. no-context baseline for downstream model evaluation

Rankings are deduplicated by canonical source identity before truncation. Tie policy, tokenization, maximum evidence tokens, and top-k values live in a versioned benchmark manifest.

Results are published at four levels:

- per case with complete ranking/evidence records
- per language and repository
- per suite and suite-native slice
- a dashboard summary that never blends unlike metrics

Common operational metrics are index build time, incremental refresh time, index bytes, p50/p95 query latency, evidence bytes/tokens, answer rate, unsupported rate, invalid rate, unresolved rate, ambiguity rate, and graph staleness signals.

Thresholds are calibrated on a declared development partition. Official evaluation labels cannot alter retriever weights, graph expansion, top-k, or release tolerances. A released threshold change requires a new contract version.

## 6. Execution lanes and publication

### Fixture lane — every PR

Small repository-owned fixtures exercise all adapters, negative leakage cases, corrupt pins, unsupported languages, artifact recovery, and metric calculations. This lane uses no network and no model.

### Deterministic audit lane — nightly and release

Runs the full pinned public corpus intersection. It may fetch missing source bytes, but scoring itself requires no model service or credential. A release report must include exact denominators and hashes.

### Model-assisted lane — scheduled advisory

Consumes frozen deterministic rankings read-only. Model, prompt, tokenizer, seed, request budget, and cache identity are pinned. Failures remain `model_pending` or `model_failed`; they cannot erase or reorder deterministic artifacts.

A deterministic report may be published after its own gate passes. A full report additionally requires complete passing model results.

In [ ]:
stateDiagram-v2
    [*] --> Prepared
    Prepared --> Frozen: freeze
    Frozen --> DeterministicScored: score_deterministic
    DeterministicScored --> ModelScored: score_model

    note right of Prepared
      @spec CODE-EVAL-ARTIFACT-LIFECYCLE
      @type ArtifactEvent = enum[freeze, score_deterministic, score_model]
      @input event: ArtifactEvent
      @state-var ranking_frozen: Bool
      @state-var deterministic_scored: Bool
      @state-var model_scored: Bool
      @requires INIT: ranking_frozen = false and deterministic_scored = false and model_scored = false
      @state Prepared
      @invariant ORDERED: (ranking_frozen = true or deterministic_scored = false) and (deterministic_scored = true or model_scored = false)
    end note

    note right of Frozen
      @state Frozen
      @transition FREEZE
      @from Prepared
      @to Frozen
      @event event = freeze
      @guard ranking_frozen = false and deterministic_scored = false and model_scored = false
      @update ranking_frozen' = true
      @update deterministic_scored' = deterministic_scored
      @update model_scored' = model_scored
    end note

    note right of DeterministicScored
      @state DeterministicScored
      @transition SCORE_DETERMINISTIC
      @from Frozen
      @to DeterministicScored
      @event event = score_deterministic
      @guard ranking_frozen = true and deterministic_scored = false and model_scored = false
      @update ranking_frozen' = ranking_frozen
      @update deterministic_scored' = true
      @update model_scored' = model_scored
    end note

    note right of ModelScored
      @state ModelScored
      @transition SCORE_MODEL
      @from DeterministicScored
      @to ModelScored
      @event event = score_model
      @guard ranking_frozen = true and deterministic_scored = true and model_scored = false
      @update ranking_frozen' = ranking_frozen
      @update deterministic_scored' = deterministic_scored
      @update model_scored' = true
      @verify INIT_ORDERED: prove initiate ORDERED
      @verify PRESERVE_FREEZE: prove preserve ORDERED on FREEZE
      @verify PRESERVE_DETERMINISTIC: prove preserve ORDERED on SCORE_DETERMINISTIC
      @verify PRESERVE_MODEL: prove preserve ORDERED on SCORE_MODEL
    end note

## 7. Artifact and command surface

The proposed crate owns adapters and orchestration, while production ranking and graph behavior remain in their existing crates.

```text
crates/spur-code-eval/
  src/
    contract.rs
    sources.rs
    materialize.rs
    query.rs
    artifacts.rs
    metrics.rs
    repoqa.rs
    crosscodeeval.rs
    jcg.rs
    model.rs
    report.rs
    main.rs
  tests/
    contract.rs
    repoqa.rs
    crosscodeeval.rs
    jcg.rs
    lifecycle.rs
    runner.rs
  benchmarks/code_eval.toml
```

Command surface:

- `validate` — fetch/check pins, schemas, licenses, denominators, and eligibility
- `index` — materialize isolated repositories and build SPUR artifacts
- `retrieve` — run all deterministic retrievers and freeze rankings/graphs
- `score` — calculate deterministic metrics from frozen artifacts
- `model` — run optional downstream evaluations against frozen contexts
- `resume` — continue incomplete model work without mutating deterministic records
- `report` — validate checksums and render a deterministic or full report

A run directory contains `manifest.json`, `validation.json`, `rankings.jsonl`, `contexts.jsonl`, serialized call graphs, `metrics.json`, optional model cache records, logs, and checksums.

## 8. Failure handling and reproducibility

- Hash, schema, repository revision, mixed-root, duplicate identity, and artifact-checksum failures are fatal.
- Unsupported language or extractor capability is nonfatal but denominator-visible.
- A missing derived CrossCodeEval evidence set is `invalid`, never an empty relevant set.
- Retrieval timeout produces a scored failure with latency evidence; it does not disappear.
- Model credential, budget, HTTP, or incomplete-response failures remain pending/failed without labels.
- A graph-staleness mismatch invalidates affected code evidence and prevents publication.
- Frozen rankings and call graphs are content-addressed and opened read-only by scorers.
- Every report records SPUR revision and dirty state, command, platform, timings, peak RSS, index bytes, dataset/repository pins, query-policy hash, and scorer versions.

Licenses are enforced per source manifest. Datasets or repositories that cannot be redistributed are fetched at runtime and never vendored. Tiny synthetic fixtures remain repository-owned.

## 9. Testing strategy and task boundaries

Implementation follows dependency order:

1. canonical contract, source pins, eligibility, and artifact identities
2. isolated repository materialization and public SPUR query adapter
3. RepoQA, CrossCodeEval, and JCG adapters, independently testable after the contract
4. deterministic metric engine and immutable artifact lifecycle
5. fixture CLI, then full deterministic runner
6. optional model backend and native downstream scorers
7. frozen baseline calibration, release gates, and documentation

Adapter work may proceed independently after the shared contract is approved. Artifact lifecycle and query execution are shared seams and must be serialized in the implementation plan.

Required tests include source-hash rejection, cross-case isolation, target-name leakage rejection, hidden-completion leakage rejection, target span resolution, unsupported-language accounting, JCG direct/indirect matching, deterministic tie handling, immutable-ranking rejection, crash recovery, model-pending behavior, and exact metric fixtures.

Existing `crates/spur-graph/tests/semantic_benchmark.rs` remains the extractor-level micro-suite and becomes a prerequisite signal rather than being renamed or replaced.

## 10. Risks and evolution

- **CrossCodeEval derived qrels:** resolver mistakes could mislabel evidence. Mitigation: version the derivation, retain unresolved identifiers, audit a stratified sample, and publish native downstream metrics alongside retrieval metrics.
- **Partial JCG expectations:** treating annotations as exhaustive would create false precision claims. Mitigation: use the official matcher as primary and report precision-like diagnostics only for exhaustive/prohibited expectations.
- **Repository cost:** full public repositories may make PR runs too slow. Mitigation: fixture lane on PRs; complete deterministic lane nightly and before releases.
- **Model drift and expense:** external models change and fail. Mitigation: pinned model/prompt/cache identity, bounded budgets, advisory lane.
- **Language expansion:** Java/C# support is a separate capability epic, not hidden benchmark scope.
- **Threshold overfitting:** calibration is development-only, frozen, and source-separated.

Promotion from Option A to Option C requires an explicit policy change: model completion becomes a hard release dependency, both deterministic and model gates must pass, and the solver/profile evidence must be regenerated.

## 11. Acceptance criteria

The design is implemented only when:

- a separate `spur-code-eval` crate runs all three adapters through one canonical contract
- public sources and repositories are revision/hash pinned and case-isolated
- deterministic rankings and call graphs freeze before scoring or model execution
- RepoQA retrieval, CrossCodeEval derived evidence, and JCG native expectations remain separately reported
- no hidden completion, target name, or gold graph edge enters retrieval input
- unsupported and invalid denominators are exact and visible
- fixture, deterministic audit, and model-assisted lanes obey the release policy
- reports include quality, cost, latency, evidence size, answer rate, and staleness
- the four formal cells in this notebook execute with fresh matching proof evidence
- implementation verification uses `scripts/spur-cargo` and the repository's normal review gates

## 12. Proof evidence

All formal cells were preflighted against the live profile registry and executed through the native NS-Mermaid runner. Every mandatory facet matched; all 28 obligations produced their expected statuses with zero mismatch or inconclusive result.

| @spec | Cell ID | Profile | Obligations | Source hash | IR hash | Report hash |
|---|---|---|---:|---|---|---|
| `CODE-EVAL-RELEASE-POLICY` | `ce000003-2026-4a00-8b00-000000000003` | `relational_lia@1` | 7/7 | `dcda3c3fc997f327…` | `4931ea805f7c138d…` | `ea17b55d4a176ec5…` |
| `CODE-EVAL-CASE-ELIGIBILITY` | `ce000006-2026-4a00-8b00-000000000006` | `relational_lia@1` | 7/7 | `1d8dc4f96493d6d8…` | `55d2543a9278f09f…` | `656dc15f6d2bb953…` |
| `CODE-EVAL-CROSSCODE-LEAKAGE` | `ce000008-2026-4a00-8b00-000000000008` | `relational_lia@1` | 6/6 | `3fc01f35fd73a1e8…` | `7a81a3763a62f7d9…` | `eb32fb0ac483b695…` |
| `CODE-EVAL-ARTIFACT-LIFECYCLE` | `ce000011-2026-4a00-8b00-000000000011` | `state_invariant_lia@1` | 8/8 | `4b45de5c0f71db64…` | `11d013dfbbb898f0…` | `14f5c502f1d4b915…` |

Independent policy-selection evidence is persisted as `sol_ab7967975cd248d4`. The model-blocking counterfactual is `sol_f752c72299fa45f8`.

Freshness status for every formal cell: source matched, metadata matched, data matched, `proof_fresh = true`.